In [ ]:
 !pip install earthscope-cli
 !pip install boto3
 !pip install earthscope-sdk
!pip install obspy


In [ ]:
!es login

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import boto3
from botocore.config import Config
from earthscope_sdk import EarthScopeClient
import pandas as pd
import numpy as np
from datetime import datetime
import os
import glob
import shutil
import time
from obspy.clients.fdsn import Client


In [ ]:
client = EarthScopeClient()

profile = client.user.get_profile()
print(profile)

In [ ]:
creds = client.user.get_aws_credentials()
print("Credentials retrieved successfully.")

In [ ]:
# Initialize S3 Client
session = boto3.Session(
    aws_access_key_id=creds.aws_access_key_id,
    aws_secret_access_key=creds.aws_secret_access_key.get_secret_value(),
    aws_session_token=creds.aws_session_token.get_secret_value(),
)

s3_client = session.client("s3",config=Config(response_checksum_validation='when_required'))

In [ ]:
S3_ACCESS_POINT = "earthscope-mseed-res-na3mtd4fq5kz7pntcyr1uh46use2a--ol-s3"
BUCKET = S3_ACCESS_POINT
PREFIX = "miniseed/"

In [ ]:
# Get list of the networks
list_resp = s3_client.list_objects_v2(Bucket=BUCKET, Prefix=PREFIX, Delimiter="/")
nets = [c["Prefix"].split("/", 1)[1] for c in list_resp["CommonPrefixes"]]
print(nets)

In [ ]:
#Finding the Lat/Lon for each of the stations

client_iris = Client("IRIS")

network_list = [net.replace('/', '').strip() for net in nets if net.replace('/', '').strip()]

station_data = []
batch_size = 50
if 'SY' in network_list:
    network_list.remove('SY')

print(f"Starting batch download for {len(network_list)} networks...")

# Batch queries
for i in range(0, len(network_list), batch_size):
    batch = network_list[i : i + batch_size]
    net_string = ",".join(batch)

    print(f"Fetching batch {i//batch_size + 1}...")

    try:
        inventory = client_iris.get_stations(network=net_string, level="station")

        for network in inventory:
            for station in network:
                end_time = station.end_date.datetime if station.end_date else pd.Timestamp('2100-01-01')

                station_data.append({
                    'network': network.code,
                    'station': station.code,
                    'lat': station.latitude,
                    'lon': station.longitude,
                    'start_date': station.start_date.datetime,
                    'end_date': end_time
                })
    except Exception as e:
        print(f"  -> Batch skipped or encountered an error: {e}")
        pass

    time.sleep(1)

df_coords = pd.DataFrame(station_data)

if not df_coords.empty:
    df_coords['start_date'] = pd.to_datetime(df_coords['start_date'])
    df_coords['end_date'] = pd.to_datetime(df_coords['end_date'])
    print(f"\nSuccess! Retrieved coordinates for {len(df_coords)} specific station epochs.")
    print(df_coords.head())
else:
    print("\nFailed to retrieve any data. Double check your network codes.")




In [ ]:
# ==========================================
# INSTANT SPATIAL BLUEPRINT (RAM-SAFE BATCHED)
# ==========================================
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from sklearn.neighbors import BallTree
import os

print("Cleaning dirty coordinates...")
# 1. Scrub missing data that causes BallTree to crash
df_coords['lat'] = pd.to_numeric(df_coords['lat'], errors='coerce')
df_coords['lon'] = pd.to_numeric(df_coords['lon'], errors='coerce')
df_coords = df_coords.dropna(subset=['lat', 'lon']).reset_index(drop=True)

print(f"Building spatial pairs from {len(df_coords)} clean IRIS stations...")

MIN_RADIUS_KM = 60.0
MAX_RADIUS_KM = 6000.0
EARTH_RADIUS_KM = 6371.0

# Convert coordinates to radians
coords_rad = np.radians(df_coords[['lat', 'lon']].values)
records = df_coords[['network', 'station', 'lat', 'lon', 'start_date', 'end_date']].values

print("Initializing Haversine BallTree...")
tree = BallTree(coords_rad, metric='haversine')

parquet_destination = "/content/drive/MyDrive/master_pairs_final.parquet"
schema = pa.schema([
    ('net1', pa.string()), ('sta1', pa.string()),
    ('lat1', pa.float32()), ('lon1', pa.float32()),
    ('net2', pa.string()), ('sta2', pa.string()),
    ('lat2', pa.float32()), ('lon2', pa.float32()),
    ('dist_km', pa.float32())
])

writer = pq.ParquetWriter(parquet_destination, schema, compression='snappy')
valid_pairs = []

# 2. Query in RAM-safe batches instead of all at once
BATCH_SIZE = 1000
total_stations = len(coords_rad)

print("Validating temporal epochs and saving in batches...")

for i in range(0, total_stations, BATCH_SIZE):
    end_idx = min(i + BATCH_SIZE, total_stations)

    # Query just this small chunk of stations
    indices_list, distances_list = tree.query_radius(
        coords_rad[i:end_idx],
        r=(MAX_RADIUS_KM / EARTH_RADIUS_KM),
        return_distance=True
    )

    for chunk_i, (neighbors, dists) in enumerate(zip(indices_list, distances_list)):
        actual_i = i + chunk_i  # The true index of Station 1

        for j, dist_rad in zip(neighbors, dists):
            # Only process pairs once (prevents A-B and B-A duplicates)
            if actual_i < j:
                dist_km = dist_rad * EARTH_RADIUS_KM

                if dist_km >= MIN_RADIUS_KM:
                    r1, r2 = records[actual_i], records[j]

                    # Check for Temporal Overlap
                    if (r1[4] <= r2[5]) and (r2[4] <= r1[5]):
                        valid_pairs.append({
                            'net1': r1[0], 'sta1': r1[1], 'lat1': float(r1[2]), 'lon1': float(r1[3]),
                            'net2': r2[0], 'sta2': r2[1], 'lat2': float(r2[2]), 'lon2': float(r2[3]),
                            'dist_km': float(dist_km)
                        })

    # Flush RAM to Parquet
    if len(valid_pairs) >= 500_000:
        writer.write_table(pa.Table.from_pylist(valid_pairs, schema=schema))
        print(f"  -> Processed up to station {end_idx:,}/{total_stations:,}...")
        valid_pairs.clear()

# Save any remaining pairs
if valid_pairs:
    writer.write_table(pa.Table.from_pylist(valid_pairs, schema=schema))

writer.close()

file_size_mb = os.path.getsize(parquet_destination) / (1024**2)
print(f"\n✅ Success! Blueprint safely stored at {parquet_destination} ({file_size_mb:.2f} MB).")

# Trigger download
from google.colab import files
files.download(parquet_destination)

In [ ]:
# ==========================================
# REBUILD MASTER S3 KEY INVENTORY (master_df)
# ==========================================
print("\nScanning S3 bucket to build full metadata inventory (this may take a while)...")
datalist = []
paginator = s3_client.get_paginator('list_objects_v2')

for net in nets:
    network = net.strip('/')
    retry_network = True

    while retry_network:
        print(f"Scanning network: {network}...")
        network_data = []
        try:
            for page in paginator.paginate(Bucket=BUCKET, Prefix=f"miniseed/{network}/"):
                if 'Contents' in page:
                    for obj in page['Contents']:
                        key = obj['Key']
                        parts = key.split('/')
                        # Expected format: miniseed/NET/YEAR/DAY/FILE
                        if len(parts) >= 5:
                            year = parts[2]
                            yearday = parts[3]
                            station = parts[-1].split('.')[0]
                            network_data.append([network, station, year, yearday, key])
            datalist.extend(network_data)
            retry_network = False

        except Exception as e:
            error_msg = str(e)
            print(f"  -> Error scanning network {network}: {error_msg}")

            if "ExpiredToken" in error_msg or "Token expired" in error_msg or "Forbidden" in error_msg:
                print("  -> Token expired. Refreshing EarthScope AWS credentials...")
                creds = client.user.get_aws_credentials()

                session = boto3.Session(
                    aws_access_key_id=creds.aws_access_key_id,
                    aws_secret_access_key=creds.aws_secret_access_key.get_secret_value(),
                    aws_session_token=creds.aws_session_token.get_secret_value(),
                )

                s3_client = session.client("s3", config=Config(response_checksum_validation='when_required'))
                paginator = s3_client.get_paginator('list_objects_v2')
            else:
                retry_network = False

master_df = pd.DataFrame(datalist, columns=['network', 'station', 'year', 'yearday', 'dataacess_key'])
print(f"Finished S3 scan! Unmerged inventory has {len(master_df):,} records.")

print("Merging S3 inventory with EarthScope coordinates (lat/lon)...")
master_df = pd.merge(master_df, df_coords[['network', 'station', 'lat', 'lon']], on=['network', 'station'], how='inner')
print(f"Merge complete! master_df now has {len(master_df):,} fully validated spatial records.")

In [ ]:
# ==========================================
# PHYSICAL COLLAPSE & SPATIAL INDEXING
# ==========================================
import gc
from sklearn.neighbors import BallTree
import pyarrow as pa
import pyarrow.parquet as pq

print("Phase 1: Collapsing timeline for unique physical stations...")

# Create a unique day identifier for temporal intersection
master_df['day_id'] = master_df['year'].astype(str) + "_" + master_df['yearday'].astype(str)

# Group by physical identity, turning active days into a Python Set for fast O(1) intersection
station_inventory = master_df.groupby(['network', 'station', 'lat', 'lon']).agg({
    'day_id': set
}).reset_index()


total_stations = len(station_inventory)
print(f"Inventory Complete: {total_stations:,} unique physical stations found.")

# ==========================================
# SPATIAL INDEXING (BallTree)
# ==========================================
print("\nPhase 2: Running global BallTree spatial match (60km - 6000km)...")
MIN_RADIUS_KM = 60.0
MAX_RADIUS_KM = 6000.0
EARTH_RADIUS_KM = 6371.0

# Convert coordinates to radians for the Haversine metric
coords_rad = np.radians(station_inventory[['lat', 'lon']].values)
records = station_inventory[['network', 'station', 'lat', 'lon']].values
active_days_sets = station_inventory['day_id'].values

tree = BallTree(coords_rad, metric='haversine')

# Find everyone within 6000km
indices_list, distances_list = tree.query_radius(
    coords_rad,
    r=(MAX_RADIUS_KM / EARTH_RADIUS_KM),
    return_distance=True
)

# ==========================================
# TEMPORAL VALIDATION & DIRECT PARQUET EXPORT
# ==========================================
print("\nPhase 3: Validating temporal overlap and saving directly to Parquet...")


# Define the exact schema for the Parquet file to ensure types are strict and lean
schema = pa.schema([
    ('net1', pa.string()), ('sta1', pa.string()),
    ('lat1', pa.float32()), ('lon1', pa.float32()),
    ('net2', pa.string()), ('sta2', pa.string()),
    ('lat2', pa.float32()), ('lon2', pa.float32()),
    ('dist_km', pa.float32()), ('overlap', pa.int32())
])

writer = pq.ParquetWriter(parquet_destination, schema, compression='snappy')

valid_pairs = []
chunk_counter = 0
CHUNK_SIZE = 1_000_000

for i, (neighbors, dists) in enumerate(zip(indices_list, distances_list)):
    for j, dist_rad in zip(neighbors, dists):
        if i < j:
            dist_km = dist_rad * EARTH_RADIUS_KM

            if dist_km >= MIN_RADIUS_KM:
                # Calculate the intersection of the two sets of active days
                shared_days = active_days_sets[i].intersection(active_days_sets[j])

                # Only keep pairs that were active at the same time
                if len(shared_days) > 0:
                    r1, r2 = records[i], records[j]

                    valid_pairs.append({
                        'net1': r1[0], 'sta1': r1[1], 'lat1': float(r1[2]), 'lon1': float(r1[3]),
                        'net2': r2[0], 'sta2': r2[1], 'lat2': float(r2[2]), 'lon2': float(r2[3]),
                        'dist_km': float(dist_km), 'overlap': int(len(shared_days))
                    })

    # Save directly to Parquet in chunks to prevent RAM overflow
    if len(valid_pairs) >= CHUNK_SIZE:
        table = pa.Table.from_pylist(valid_pairs, schema=schema)
        writer.write_table(table)
        chunk_counter += 1
        print(f"  -> Compressed and saved chunk {chunk_counter} ({CHUNK_SIZE:,} pairs)...")
        valid_pairs.clear()

# Save the final remainder
if valid_pairs:
        table = pa.Table.from_pylist(valid_pairs, schema=schema)
        writer.write_table(table)
        chunk_counter += 1
        print(f"  -> Compressed and saved final chunk {chunk_counter} ({len(valid_pairs):,} pairs)...")

writer.close()

# Check the final file size
file_size_mb = os.path.getsize(parquet_destination) / (1024**2)
print(f"\nSuccess! Cleaned master blueprint safely stored directly to Parquet: {parquet_destination} ({file_size_mb:.2f} MB).")

In [ ]:
# ==========================================
# 1. BUILD PARTITIONED DAILY KEY INDEX
# ==========================================
import sys

print("Building the strict-schema Daily Key Index...")

# Extract only the necessary columns from our existing master_df
key_index = master_df[['network', 'station', 'year', 'yearday', 'dataacess_key']].copy()

# Optimize data types for massive memory savings
key_index['network'] = key_index['network'].astype('category')
key_index['station'] = key_index['station'].astype('category')
key_index['year'] = key_index['year'].astype('int16')
key_index['yearday'] = key_index['yearday'].astype('int16')

# Save as a partitioned Parquet dataset
index_dir = "keys_partitioned_year"
key_index.to_parquet(
    index_dir,
    engine='pyarrow',
    partition_cols=['year'],
    index=False
)

print(f"Success! Key Index safely stored at {index_dir}")

# Safely delete the massive dataframes to clear up RAM
del master_df
del key_index
gc.collect()

# ==========================================
# 2. ON-DEMAND MANIFEST GENERATOR
# ==========================================

def get_keys(net1, sta1, net2, sta2, year, key_index_path="keys_partitioned_year"):
    """
    Retrieves overlapping S3 keys for two specific stations in a given year.
    """
    try:
        # Load only the specific year partition (High speed, Low RAM)
        year_df = pd.read_parquet(
            key_index_path,
            filters=[('year', '==', int(year))]
        )
    except Exception:
        return pd.DataFrame() # Returns empty if year doesn't exist

    # Filter for Station 1 and Station 2
    s1_data = year_df[(year_df['network'] == net1) & (year_df['station'] == sta1)]
    s2_data = year_df[(year_df['network'] == net2) & (year_df['station'] == sta2)]

    if s1_data.empty or s2_data.empty:
        return pd.DataFrame()

    # Inner Join to find overlapping days
    manifest = pd.merge(
        s1_data[['yearday', 'dataacess_key']],
        s2_data[['yearday', 'dataacess_key']],
        on='yearday',
        suffixes=('_1', '_2')
    )

    # Clean up columns
    manifest = manifest.rename(columns={
        'dataacess_key_1': 'key1',
        'dataacess_key_2': 'key2'
    }).sort_values('yearday').reset_index(drop=True)

    return manifest

# --- TEST IT OUT ---
download_list = get_keys('IU', 'ANMO', 'IU', 'RSSD', 2024)

if not download_list.empty:
    print(f"\nSuccess! Found {len(download_list)} overlapping days for test pair.")
    print(download_list.head())
else:
    print("\nNo overlapping days found for test pair.")

In [ ]:
# ==========================================
# EXPORT METADATA TO LOCAL MACHINE
# ==========================================
import shutil
from google.colab import files

print("Zipping the partitioned key index folder...")
# This creates a file named 'keys_partitioned_year.zip'
shutil.make_archive("keys_partitioned_year", 'zip', "keys_partitioned_year")

print("Triggering browser download for Master Blueprint...")
files.download("master_pairs_final.parquet")

print("Triggering browser download for Key Index ZIP...")
files.download("keys_partitioned_year.zip")

print("Downloads initiated! (Check your browser's download manager)")